<a href="https://colab.research.google.com/github/nisaa783/scikit-learn-cookbook_Anisa-Khasanah/blob/main/Chapter_8_Tree_Based_Algorithms_and_Ensemble_Methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Pendahuluan dan Algoritma Berbasis Pohon (Decision Trees)
Algoritma berbasis pohon keputusan (Decision Trees) memetakan keputusan dalam bentuk struktur pohon bercabang (mirip diagram alir), mulai dari simpul akar (root node), simpul internal (internal node yang menguji kondisi pada suatu fitur), hingga simpul daun (leaf node yang memuat hasil prediksi kelas atau nilai).

Kelebihan utama Decision Tree adalah mudah diinterpretasikan secara visual dan logis, tidak memerlukan feature scaling, serta mampu menangkap hubungan non-linear antar fitur. Namun, satu pohon tunggal cenderung mudah mengalami overfitting karena dapat membuat cabang terlalu spesifik pada data latih.

## 2. Metode Ensemble: Bagging dan Boosting
Untuk mengatasi kelemahan Decision Tree tunggal, kita menggabungkan banyak model dasar (base learners) menggunakan teknik Ensemble Methods:
Bagging (Bootstrap Aggregating) & Random Forest: Melatih banyak

*   Decision Tree secara paralel pada sampel acak hasil bootstrap dari data latih, lalu menggabungkan hasilnya (voting mayoritas untuk klasifikasi atau rata-rata untuk regresi). Random Forest menambahkan variasi dengan memilih subset fitur secara acak pada setiap pemecahan simpul (split), sehingga mengurangi korelasi antar pohon dan menekan overfitting.
*   Boosting & Gradient Boosting: Membangun pohon secara berurutan (sekuensial), di mana setiap pohon baru dilatih untuk memperbaiki kesalahan (residual atau gradien loss) dari pohon-pohon sebelumnya.

## 3. Contoh Program Penerapan 1: Klasifikasi dengan Decision Tree dan Random Forest
Contoh program ini membandingkan performa DecisionTreeClassifier tunggal dengan RandomForestClassifier pada dataset load_breast_cancer.


In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# 1. Memuat dataset klasifikasi kanker payudara
data = load_breast_cancer()
X, y = data.data, data.target

# Membagi data menjadi data latih (75%) dan data uji (25%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# 2. Melatih Decision Tree tunggal
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

# 3. Melatih Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# 4. Membandingkan akurasi kedua model
print(f"Akurasi Decision Tree (Data Uji): {accuracy_score(y_test, dt_pred):.4f}")
print(f"Akurasi Random Forest (Data Uji): {accuracy_score(y_test, rf_pred):.4f}")
print("\nLaporan Klasifikasi Random Forest:")
print(classification_report(y_test, rf_pred, target_names=data.target_names))

Akurasi Decision Tree (Data Uji): 0.9441
Akurasi Random Forest (Data Uji): 0.9580

Laporan Klasifikasi Random Forest:
              precision    recall  f1-score   support

   malignant       0.96      0.92      0.94        53
      benign       0.96      0.98      0.97        90

    accuracy                           0.96       143
   macro avg       0.96      0.95      0.95       143
weighted avg       0.96      0.96      0.96       143



## 4. Contoh Program Penerapan 2: Gradient Boosting dan Feature Importance
Contoh program kedua ini menggunakan GradientBoostingClassifier serta mengekstrak skor kepentingan fitur (feature importances) untuk mengetahui fitur mana yang paling berkontribusi dalam proses pengambilan keputusan pohon.

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Memuat dataset
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# 2. Inisialisasi dan melatih Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_model.fit(X_train, y_train)

# 3. Evaluasi akurasi dan ROC-AUC
y_pred = gb_model.predict(X_test)
y_prob = gb_model.predict_proba(X_test)[:, 1]

print(f"Akurasi Gradient Boosting: {accuracy_score(y_test, y_pred):.4f}")
print(f"Skor ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

# 4. Menampilkan 5 fitur terpenting berdasarkan model
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': gb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\n5 Fitur Terpenting (Top 5 Feature Importances):")
print(importance_df.head(5).to_string(index=False))

Akurasi Gradient Boosting: 0.9580
Skor ROC-AUC: 0.9925

5 Fitur Terpenting (Top 5 Feature Importances):
             Feature  Importance
        worst radius    0.558938
worst concave points    0.131888
     worst perimeter    0.127710
       texture error    0.037132
       worst texture    0.032110
